# Star Wars: Galaxy of Heroes — Onboarding & Retention: SQL Analysis

This notebook is the SQL analysis layer of the project, run with **SQLite**
directly inside Python so every query and its result table is visible right
here on GitHub — no need to clone the repo or run anything to see the output.

**Context:** *Star Wars: Galaxy of Heroes* (SWGOH) is a real free-to-play
mobile collectible RPG developed by EA Capital Games and published by
Electronic Arts (released November 24, 2015), in which players assemble
squads of Star Wars characters, fight turn-based battles, gear up and level
up heroes, and take part in guild-based Territory Wars and live events. It
is used here purely as illustrative, real-world context for a portfolio
analytics exercise.

**Data note:** All player-level data queried below (`data/players.csv`,
`data/player_events.csv`, `data/sessions.csv`) is **synthetic — generated
for this project**. It does not represent actual SWGOH telemetry, and this
project is not affiliated with, endorsed by, or representative of
Electronic Arts, EA Capital Games, Lucasfilm, or Disney.

## What this notebook does

1. Loads the three CSV datasets and builds a SQLite database from them (so
   the notebook is fully self-contained — it doesn't depend on a pre-built
   `.sqlite` file).
2. Runs the funnel, retention, and activity SQL queries used by the rest of
   the project (documented in full in `sql/queries.sql`).
3. Prints each result as a table, with a short note on what it shows.


In [1]:
import sqlite3
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

DATA_DIR = '../data'
DB_PATH = ':memory:'   # built fresh from the CSVs every run — nothing to download separately

conn = sqlite3.connect(DB_PATH)


## 1. Load the datasets and build the SQLite database

The three CSVs mirror how most mobile games log data: one row per player, one row per behavioral event, one row per play session.

In [2]:
players = pd.read_csv(f'{DATA_DIR}/players.csv')
player_events = pd.read_csv(f'{DATA_DIR}/player_events.csv')
sessions = pd.read_csv(f'{DATA_DIR}/sessions.csv')

players.to_sql('players', conn, index=False, if_exists='replace')
player_events.to_sql('player_events', conn, index=False, if_exists='replace')
sessions.to_sql('sessions', conn, index=False, if_exists='replace')

conn.execute('CREATE INDEX idx_events_player ON player_events(player_id)')
conn.execute('CREATE INDEX idx_events_name ON player_events(event_name)')
conn.execute('CREATE INDEX idx_sessions_player ON sessions(player_id)')

print(f'players:        {len(players):,} rows')
print(f'player_events:  {len(player_events):,} rows')
print(f'sessions:       {len(sessions):,} rows')


players:        50,000 rows
player_events:  663,019 rows
sessions:       116,854 rows


## 2. Total players

The base every funnel and retention rate below is measured against.

In [3]:
pd.read_sql_query('''
    SELECT COUNT(DISTINCT player_id) AS total_players
    FROM players
''', conn)


,total_players
0,50000


## 3. Tutorial completion rate

Of players who ever start the tutorial, how many finish it?

In [4]:
pd.read_sql_query('''
    WITH starters AS (
        SELECT DISTINCT player_id FROM player_events WHERE event_name = 'tutorial_start'
    ),
    completers AS (
        SELECT DISTINCT player_id FROM player_events WHERE event_name = 'tutorial_complete'
    )
    SELECT
        (SELECT COUNT(*) FROM starters)   AS tutorial_starters,
        (SELECT COUNT(*) FROM completers) AS tutorial_completers,
        ROUND(100.0 * (SELECT COUNT(*) FROM completers)
                     / (SELECT COUNT(*) FROM starters), 2) AS completion_rate_pct
''', conn)


,tutorial_starters,tutorial_completers,completion_rate_pct
0,45086,15374,34.1


## 4. First-battle completion rate

Of players who reach their first battle, how many complete it?

In [5]:
pd.read_sql_query('''
    WITH reached AS (
        SELECT DISTINCT player_id FROM player_events WHERE event_name = 'first_battle'
    ),
    completed AS (
        SELECT DISTINCT player_id FROM player_events WHERE event_name = 'battle_complete'
    )
    SELECT
        (SELECT COUNT(*) FROM reached)   AS first_battle_reached,
        (SELECT COUNT(*) FROM completed) AS first_battle_completed,
        ROUND(100.0 * (SELECT COUNT(*) FROM completed)
                     / (SELECT COUNT(*) FROM reached), 2) AS battle_completion_rate_pct
''', conn)


,first_battle_reached,first_battle_completed,battle_completion_rate_pct
0,14034,9945,70.86


## 5. Player activity: sessions, average duration, active days

Per-player engagement depth — the raw material for the segmentation and correlation analysis in `src/analyze.py`.

In [6]:
activity = pd.read_sql_query('''
    SELECT
        p.player_id,
        COUNT(s.session_id)                    AS num_sessions,
        ROUND(AVG(s.session_duration), 2)      AS avg_session_minutes,
        COUNT(DISTINCT DATE(s.session_start))  AS active_days
    FROM players p
    LEFT JOIN sessions s ON p.player_id = s.player_id
    GROUP BY p.player_id
''', conn)

print(f'{len(activity):,} players')
activity.describe()


50,000 players


,num_sessions,avg_session_minutes,active_days
count,50000.000000,50000.000000,50000.000000
mean,2.337080,11.634654,2.162920
std,1.188581,3.753931,0.979365
min,1.000000,1.000000,1.000000
25%,1.000000,9.140000,1.000000
50%,2.000000,11.690000,2.000000
75%,3.000000,14.180000,3.000000
max,11.000000,27.630000,8.000000


## 6. D1 retention

What share of all players returned exactly one day after install?

In [7]:
pd.read_sql_query('''
    WITH d1_active AS (
        SELECT DISTINCT s.player_id
        FROM sessions s
        JOIN players p ON p.player_id = s.player_id
        WHERE DATE(s.session_start) = DATE(p.install_date, '+1 day')
    )
    SELECT
        COUNT(DISTINCT p.player_id) AS total_players,
        COUNT(DISTINCT d.player_id) AS d1_returned,
        ROUND(100.0 * COUNT(DISTINCT d.player_id) / COUNT(DISTINCT p.player_id), 2) AS d1_retention_pct
    FROM players p
    LEFT JOIN d1_active d ON p.player_id = d.player_id
''', conn)


,total_players,d1_returned,d1_retention_pct
0,50000,23449,46.9


## 7. D7 retention

Only players with at least 7 days of history are eligible — otherwise a recent install would be counted as \"churned\" before Day 7 even arrives.

In [8]:
pd.read_sql_query('''
    WITH eligible AS (
        SELECT player_id, install_date FROM players
        WHERE DATE(install_date, '+7 day') <= DATE('2026-09-22')
    ),
    d7_active AS (
        SELECT DISTINCT s.player_id
        FROM sessions s
        JOIN eligible e ON e.player_id = s.player_id
        WHERE DATE(s.session_start) = DATE(e.install_date, '+7 day')
    )
    SELECT
        COUNT(DISTINCT e.player_id) AS eligible_players,
        COUNT(DISTINCT d.player_id) AS d7_returned,
        ROUND(100.0 * COUNT(DISTINCT d.player_id) / COUNT(DISTINCT e.player_id), 2) AS d7_retention_pct
    FROM eligible e
    LEFT JOIN d7_active d ON e.player_id = d.player_id
''', conn)


,eligible_players,d7_returned,d7_retention_pct
0,50000,15841,31.68


## 8. Retention by onboarding behavior — the key comparison

This is the query that drives the project's central finding: does completing the tutorial actually predict who comes back?

In [9]:
pd.read_sql_query('''
    WITH tut_complete AS (
        SELECT DISTINCT player_id FROM player_events WHERE event_name = 'tutorial_complete'
    ),
    eligible AS (
        SELECT player_id, install_date FROM players
        WHERE DATE(install_date, '+7 day') <= DATE('2026-09-22')
    ),
    d7_active AS (
        SELECT DISTINCT s.player_id
        FROM sessions s JOIN eligible e ON e.player_id = s.player_id
        WHERE DATE(s.session_start) = DATE(e.install_date, '+7 day')
    )
    SELECT
        CASE WHEN tc.player_id IS NOT NULL THEN 'Tutorial Completed' ELSE 'Tutorial Not Completed' END AS cohort,
        COUNT(DISTINCT e.player_id) AS players,
        ROUND(100.0 * COUNT(DISTINCT d.player_id) / COUNT(DISTINCT e.player_id), 2) AS d7_retention_pct
    FROM eligible e
    LEFT JOIN tut_complete tc ON tc.player_id = e.player_id
    LEFT JOIN d7_active d ON d.player_id = e.player_id
    GROUP BY cohort
''', conn)


,cohort,players,d7_retention_pct
0,Tutorial Completed,15374,52.15
1,Tutorial Not Completed,34626,22.59


## 9. Retention by first-battle completion

The same comparison, but for the deeper milestone — completing the first battle rather than just the tutorial script.

In [10]:
pd.read_sql_query('''
    WITH battle_complete AS (
        SELECT DISTINCT player_id FROM player_events WHERE event_name = 'battle_complete'
    ),
    eligible AS (
        SELECT player_id, install_date FROM players
        WHERE DATE(install_date, '+7 day') <= DATE('2026-09-22')
    ),
    d7_active AS (
        SELECT DISTINCT s.player_id
        FROM sessions s JOIN eligible e ON e.player_id = s.player_id
        WHERE DATE(s.session_start) = DATE(e.install_date, '+7 day')
    )
    SELECT
        CASE WHEN bc.player_id IS NOT NULL THEN 'First Battle Completed' ELSE 'First Battle Not Completed' END AS cohort,
        COUNT(DISTINCT e.player_id) AS players,
        ROUND(100.0 * COUNT(DISTINCT d.player_id) / COUNT(DISTINCT e.player_id), 2) AS d7_retention_pct
    FROM eligible e
    LEFT JOIN battle_complete bc ON bc.player_id = e.player_id
    LEFT JOIN d7_active d ON d.player_id = e.player_id
    GROUP BY cohort
''', conn)


,cohort,players,d7_retention_pct
0,First Battle Completed,9945,61.73
1,First Battle Not Completed,40055,24.22


## Key takeaway from the SQL layer

Completing the first battle is associated with a larger Day-7 retention gap
than completing the tutorial alone — the same signal the deeper Python
analysis (`src/analyze.py`, and the segmentation/correlation work in the
final report) builds on to make the case for re-sequencing onboarding
around a player's first real win rather than the tutorial script itself.

These are observed associations in the dataset, not proof of causation —
see `output/player_analytics_report.html` for the full analysis and the
proposed A/B test that would test this causally.
